# English → Spanish Neural Machine Translation
## Custom Transformer in PyTorch — Track A

**Natural Language Processing — Final Project**

This notebook runs the complete pipeline: data preparation, model construction,
training, evaluation, and export. The model implementation itself lives in the
GitHub repository (`src/`) and is cloned in Section 0 — Colab is used only as a
GPU runtime, not as the place the code is written.

| Section | Milestone | Runtime |
|---|---|---|
| 0 | Setup — GPU, Drive, clone | ~1 min |
| 1 | Data pipeline, tokenizer, DataLoader | ~5 min |
| 2 | Model construction and mask verification | ~1 min |
| 3 | Training both capacity presets | ~50 min |
| 4 | Held-out test evaluation and error analysis | ~10 min |
| 5 | Length-penalty tuning | ~15 min |
| 6 | Export for local inference | ~1 min |

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Checkpoints are written to Google Drive after every epoch. If the session is
killed, re-running the training cell resumes from the last completed epoch
rather than starting over.

---
# 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

### 0.1 Mount Drive

Colab wipes local storage when a session ends. Checkpoints must live in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/nmt-en-es'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
!mkdir -p "{CKPT_DIR}"
print('checkpoints ->', CKPT_DIR)

### 0.2 Clone the project

All model code is version-controlled. Colab clones it rather than defining it
inline, so the code that trains here is byte-identical to the code that runs
locally.

In [ ]:
!git clone https://github.com/halxbrown/nmt-en-es.git /content/nmt-en-es
%cd /content/nmt-en-es
!ls

In [ ]:
!pip install -q sentencepiece sacrebleu rouge-score 'datasets>=3.0'
# torch is preinstalled and CUDA-matched -- do not reinstall it.

---
# 1. Milestone 1 — Data Pipeline

Downloads OPUS-100 `en-es`, applies eight cleaning filters, samples 100,000
pairs, trains a joint 16k SentencePiece BPE vocabulary, and builds the
DataLoader with dynamic padding and length bucketing.

**What to look for in the output:**
- The filter table (16.46% dropped) — Table 1 of the report
- Fertility 1.49 on both sides, UNK rate 0.0000%
- Padding efficiency 22.8% → 78.0% (3.42× fewer wasted attention cells)
- Four mask verification PASSes

In [ ]:
!python run_milestone1.py --force

---
# 2. Milestone 2 — Model & Mask Verification

Builds the Transformer and verifies correctness **behaviourally** rather than
by inspection. A masking bug degrades translation quality without raising an
exception, so structural assertions alone are not sufficient.

**What to look for:**
- 37,619,712 parameters; embedding and output projection share storage
- Initial loss 10.27 vs ln(16000)=9.68 — the excess is the tied-embedding copy
  bias, explained in Section 3.3 of the report
- **Causal test: `0.000e+00`** before the corrupted position — no information
  flows backwards in time
- **Padding test: ~`5.7e-06`** — GPU float non-associativity, four orders of
  magnitude below a real leak
- Single-batch overfit: loss falls 10.27 → ~1.4

In [ ]:
!python run_milestone2.py --steps 60

---
# 3. Milestone 3 — Training

Trains two capacity presets with **identical regularization**, so the ablation
isolates capacity rather than confounding it with dropout.

| | base | small |
|---|---:|---:|
| Layers | 4 + 4 | 3 + 3 |
| `d_ff` | 2048 | 1024 |
| Parameters | 37.6M | 24.0M |
| Dropout | 0.2 | 0.2 |

AdamW, peak LR 5e-4, 2000-step warmup then cosine decay, label smoothing 0.1,
fp16 autocast, early stopping patience 3.

**~50 minutes.** Checkpoints save to Drive every epoch — if the session dies,
just re-run this cell.

In [ ]:
!python run_milestone3.py --preset both --epochs 20 --checkpoint-dir "{CKPT_DIR}"

### 3.1 Decoder correctness

Three properties BLEU cannot establish. A subtly broken beam search still
produces fluent output and a plausible score.

1. `beam(k=1, α=0)` must reproduce greedy **token-for-token** — with one beam
   there is nothing to compare, so it must degenerate to argmax
2. `beam(k=5)` must find sequences of **higher model log-probability** than
   greedy — that is beam search's entire purpose
3. Mean hypothesis length must be **non-decreasing in α**

In [ ]:
!python verify_decoding.py --preset base  --checkpoint-dir "{CKPT_DIR}"
!python verify_decoding.py --preset small --checkpoint-dir "{CKPT_DIR}"

### 3.2 Learning curves

Neither model triggered early stopping — validation loss improved monotonically
through epoch 19. Note that the learning rate reached its cosine floor exactly
as the curves flattened, so convergence cannot be distinguished from the
schedule ending.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, preset in zip(axes, ['base', 'small']):
    p = Path(CKPT_DIR) / preset / 'history.json'
    if not p.exists():
        ax.set_title(f'{preset}: no history found')
        continue
    h = json.loads(p.read_text())
    ep = [r['epoch'] for r in h]
    ax.plot(ep, [r['train_loss'] for r in h], 'o-', label='train')
    ax.plot(ep, [r['val_loss'] for r in h], 's-', label='validation')
    best = min(h, key=lambda r: r['val_loss'])
    ax.axvline(best['epoch'], ls='--', c='grey', lw=1)
    ax.annotate(f"best epoch {best['epoch']}\nval {best['val_loss']:.3f}",
                (best['epoch'], best['val_loss']),
                textcoords='offset points', xytext=(10, 20), fontsize=9)
    ax.set_title(f"{preset}  ({len(h)} epochs)")
    ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
Path('artifacts').mkdir(exist_ok=True)
plt.savefig('artifacts/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 4. Milestone 4 — Held-Out Test Evaluation

Milestone 3's numbers came from the **validation** set, which selected the
checkpoint, so they are optimistically biased. Everything here runs on `test`,
untouched until now.

Includes paired-bootstrap significance testing (Koehn, 2004), BLEU bucketed by
sentence length, heuristic error categorisation, and cross-attention maps.

**What to look for:**
- Test BLEU slightly below validation — that gap is the selection bias
- Every comparison carries a 95% CI and p-value
- Length ratio ≈ 0.92 in *every* bucket — a uniform brevity bias

In [ ]:
!python run_milestone4.py --preset both --checkpoint-dir "{CKPT_DIR}"

### 4.1 Attention maps

Cross-attention from the final decoder layer, averaged over heads. The diagonal
band is emergent word alignment — nothing in the loss function asked for it.

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('artifacts/attention_*.png')):
    print(p)
    display(Image(filename=p, width=780))

### 4.2 Worst-scoring sentences

Ranked by sentence chrF. Tags are heuristic hints; the report quotes sentences
that were read individually.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('artifacts/failures_base.md', encoding='utf-8').read()))

---
# 5. Length-Penalty Tuning

Section 4.5 of the report. The model under-generates by ~8% at every sentence
length, and BLEU penalises that through the brevity penalty.

α is swept on **validation** and only the winning value is applied to test.
Tuning a decoding hyperparameter by test BLEU would contaminate the test set
exactly as selecting a checkpoint on test would.

**What to look for:** the validation curve rises monotonically across the whole
range, but the spread above α≈1.8 is smaller than the sampling noise — and the
validation argmax transfers *worse* to test than a smaller value does. A
monotonically improving tuning curve is not evidence that the largest value is
best.

In [ ]:
!python sweep_length_penalty.py --preset base --alphas 0.6 1.0 1.4 1.8 2.2 2.6 3.0 3.5 4.0 --checkpoint-dir "{CKPT_DIR}"

In [ ]:
!python sweep_length_penalty.py --preset small --alphas 0.6 1.0 1.4 1.8 2.2 --checkpoint-dir "{CKPT_DIR}"

---
# 6. Export for Local Inference

`best.pt` carries AdamW's two moment tensors per parameter plus scheduler state
and epoch history — roughly 340 MB, where the weights alone are 113 MB.
Inference needs none of it.

Download `base_inference.pt` from Drive into the local `models/` folder, then
run `python translate.py` in VS Code.

In [ ]:
!python export_model.py --preset base  --checkpoint-dir "{CKPT_DIR}"
!python export_model.py --preset small --checkpoint-dir "{CKPT_DIR}"
!cp "{CKPT_DIR}"/base/base_inference.pt "{DRIVE_ROOT}/"
!cp "{CKPT_DIR}"/small/small_inference.pt "{DRIVE_ROOT}/"
print()
!ls -lh "{DRIVE_ROOT}"/*_inference.pt

### 6.1 Sanity check — translate in Colab

The same `translate.py` that runs locally. Training and inference share one
implementation, so there is no separate deployment path that could drift from
the evaluated one.

In [ ]:
CKPT = f"{CKPT_DIR}/base/base_inference.pt"
!python translate.py --preset base --checkpoint "{CKPT}" --text "Where is the train station?" --compare
print()
!python translate.py --preset base --checkpoint "{CKPT}" --text "I have been waiting here for over an hour."

---
# 7. Save Everything to Drive

Colab's local disk is wiped at session end. These artifacts are what the report
is written from.

In [ ]:
!cp -r artifacts "{DRIVE_ROOT}/"
!ls "{DRIVE_ROOT}/artifacts"

---
# 8. Consolidated Results

Every headline number in one place, for the report's Results section.

In [ ]:
import json
from pathlib import Path

A = Path('artifacts')

m4 = json.loads((A / 'milestone4_results.json').read_text())
print("=" * 68)
print("  FINAL TEST-SET RESULTS (2,000 held-out sentences)")
print("=" * 68)
print(f"{'model':<8}{'params':>12}{'greedy':>9}{'beam':>9}{'gain':>8}{'chrF':>8}{'ROUGE-L':>9}")
print("-" * 63)
for k in ('base', 'small'):
    if k not in m4:
        continue
    v = m4[k]
    g, b = v['decoding']['greedy'], v['decoding']['beam']
    print(f"{k:<8}{v['parameters']:>12,}{g['bleu']:>9}{b['bleu']:>9}"
          f"{b['bleu'] - g['bleu']:>+8.2f}{b['chrf']:>8}{b.get('rougeL', 0):>9}")

print("\nSignificance (paired bootstrap, 1000 resamples):")
for k in ('base', 'small'):
    if k in m4:
        s = m4[k]['beam_vs_greedy']
        print(f"  {k:<6} beam-greedy {s['delta']:+.2f}  "
              f"CI [{s['ci95_low']:+.2f},{s['ci95_high']:+.2f}]  p={s['p_value']}")
if 'base_vs_small' in m4:
    s = m4['base_vs_small']
    print(f"  base - small {s['delta']:+.2f}  "
          f"CI [{s['ci95_low']:+.2f},{s['ci95_high']:+.2f}]  p={s['p_value']}")

for preset in ('base', 'small'):
    f = A / f'length_penalty_sweep_{preset}.json'
    if not f.exists():
        continue
    d = json.loads(f.read_text())
    t = d['test_comparison']
    print(f"\nLength penalty — {preset}: alpha={d['chosen_alpha']} "
          f"(selected on validation)")
    print(f"  {t['bleu_a']} -> {t['bleu_b']}   {t['delta']:+.2f}  "
          f"CI [{t['ci95_low']:+.2f},{t['ci95_high']:+.2f}]  p={t['p_value']}")

print("\nBLEU by source length (base, beam):")
if 'base' in m4:
    print(f"  {'bucket':<10}{'n':>7}{'BLEU':>8}{'len ratio':>11}")
    for b in m4['base']['length_buckets']:
        print(f"  {b['bucket']:<10}{b['n']:>7}{b['bleu']:>8}{b['len_ratio']:>11}")

print("\nError categories (base, beam):")
if 'base' in m4:
    for k, v in m4['base']['error_categories'].items():
        print(f"  {k:<22}{v['n']:>6}{v['pct']:>8}%")
print()
print("sacreBLEU signature:",
      m4.get('base', {}).get('decoding', {}).get('beam', {}).get('signature', 'n/a'))

---
## Next: local inference

The model is trained and exported. Download `base_inference.pt` from Drive into
`models/` in the local repository, then:

```bash
python setup_local.py     # verify the install
python translate.py       # interactive English -> Spanish
```

Training and inference share the same `src/model.py` and `src/decode.py`, so
the test BLEU reported above describes exactly the code path that runs locally.